<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/03_search/search_ranking_quality_mrr_ndcg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Ranking Quality with MRR and nDCG

## Objective
This notebook evaluates the quality of semantic search rankings
using Mean Reciprocal Rank (MRR) and Normalized Discounted
Cumulative Gain (nDCG).

The goal is to measure how early and how correctly relevant
documents appear in ranked search results.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
import pandas as pd
import numpy as np

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
query = "learn machine learning basics"

documents = [
    "Machine learning tutorials for beginners",
    "Introduction to deep learning",
    "Python programming fundamentals",
    "Natural language processing with transformers",
    "Football match highlights",
    "Top travel destinations"
]

# Graded relevance: 3 = highly relevant, 2 = relevant, 1 = weak, 0 = not relevant
relevance = {
    "Machine learning tutorials for beginners": 3,
    "Introduction to deep learning": 2,
    "Python programming fundamentals": 1,
    "Natural language processing with transformers": 1,
    "Football match highlights": 0,
    "Top travel destinations": 0
}

In [7]:
query_emb = model.encode(query)
doc_embs = model.encode(documents)

scores = cosine_similarity([query_emb], doc_embs)[0]

df = pd.DataFrame({
    "Document": documents,
    "Similarity Score": np.round(scores, 4),
    "Relevance": [relevance[d] for d in documents]
}).sort_values(by="Similarity Score", ascending=False)

df

,Document,Similarity Score,Relevance
0,Machine learning tutorials for beginners,0.8806,3
1,Introduction to deep learning,0.5455,2
2,Python programming fundamentals,0.3898,1
3,Natural language processing with transformers,0.2065,1
4,Football match highlights,0.0823,0
5,Top travel destinations,0.0293,0


In [8]:
def mean_reciprocal_rank(df):
    for idx, rel in enumerate(df["Relevance"]):
        if rel > 0:
            return 1 / (idx + 1)
    return 0

mrr = mean_reciprocal_rank(df)
mrr

1.0

In [9]:
y_true = np.array([df["Relevance"].values])
y_score = np.array([df["Similarity Score"].values])

ndcg = ndcg_score(y_true, y_score)
ndcg

np.float64(1.0)

## Observations

- MRR captures how quickly the first relevant document appears
- nDCG evaluates the overall ranking quality using graded relevance
- High nDCG indicates good ordering, not just retrieval

## Key Insight
Ranking quality matters as much as retrieval accuracy,
especially in user-facing search and RAG systems.